# Full Evaluation — Groq API (150 Queries)
### MSc Dissertation 7PAM2002 — Graph-Constrained Vector Retrieval for Mitigating Structural Failure Modes in Naive RAG

---

**Purpose**  
This notebook runs the full 150-query evaluation across all three pipeline variants:

| Variant | Description | What changes |
|---------|-------------|--------------|
| **V0** | Naive RAG baseline | Fixed-window chunking (256 tokens, 32 overlap) + cosine similarity |
| **V1** | Structure-aware chunking | Paragraph-boundary chunking + cosine similarity |
| **V2** | Graph-constrained retrieval | Paragraph-boundary chunking + proximity-weighted graph re-ranking |

V0 vs V1 isolates the **chunking effect**.  
V1 vs V2 isolates the **graph constraint effect**.  
V0 vs V2 shows the **combined effect** of both changes.

---

**Prerequisites**
- All data artefacts must exist in `data/` (built by pilot notebooks):
  - `queries.json`, `v0_chunks.json`, `v0_index.faiss`
  - `v1_chunks.json`, `v1_index.faiss`, `v2_graph.pkl`
- A `.env` file in the project root with your Groq API key
- Run: `pip install -r requirements.txt`

---

**Resumption**  
If interrupted, re-run from Cell 1. The notebook checks which queries already have results and resumes automatically. No API calls are duplicated.


## Cell 1 — Install Dependencies

In [ ]:
# Install all required packages
# groq and langchain-groq replace ollama for the full evaluation run
# python-dotenv loads the API key from .env without hardcoding it

!pip install -q \
    groq \
    langchain-groq \
    python-dotenv \
    datasets \
    sentence-transformers \
    faiss-cpu \
    ragas \
    langchain \
    langchain-community \
    networkx \
    numpy

print("All packages installed.")


## Cell 2 — Load Groq API Key

The API key is stored in a `.env` file in the project root — it is **never hardcoded** in this notebook and is excluded from version control via `.gitignore`.

**Setup steps (run once):**
1. Copy `.env.example` to `.env`
2. Open `.env` and add your Groq API key
3. Get a free key at https://console.groq.com (no credit card required)


In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file in the project root
# This reads GROQ_API_KEY without exposing it in the notebook
load_dotenv()

api_key = os.environ.get("GROQ_API_KEY")

if not api_key:
    raise EnvironmentError(
        "GROQ_API_KEY not found.\n"
        "Steps to fix:\n"
        "  1. Copy .env.example to .env in the project root\n"
        "  2. Add your Groq API key to .env\n"
        "  3. Get a free key at https://console.groq.com"
    )

# Print only first 8 characters as a visual confirmation — never print the full key
print(f"API key loaded successfully: {api_key[:8]}...")


## Cell 3 — Imports

In [ ]:
import json
import os
import time
import pickle
import traceback
from datetime import datetime
from collections import defaultdict

import numpy as np
import faiss
import networkx as nx
from datasets import Dataset
from sentence_transformers import SentenceTransformer

# Groq client for generation — replaces ollama.chat() from pilot notebooks
from groq import Groq

# LangChain + Groq wrapper for RAGAS evaluation
# RAGAS needs a LangChain-compatible LLM interface
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas import evaluate, RunConfig
from ragas.metrics import context_recall, context_precision, faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

print("All imports OK")
print(f"NetworkX version: {nx.__version__}")


## Cell 4 — Configuration

All parameters are declared here in one place. Nothing is hardcoded inside functions.  
Parameters marked **FIXED** are identical across V0, V1, and V2 — changing them would invalidate the controlled comparison.


In [ ]:
# ── Models — FIXED across all variants
EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"  # 384-dim
GROQ_MODEL   = "llama-3.1-8b-instant"                     # Llama 3.1 8B via Groq
TEMPERATURE  = 0                                           # greedy — fully deterministic

# ── Retrieval — FIXED for V0 and V1
TOP_K_STANDARD = 5    # context chunks passed to LLM in V0 and V1

# ── V2-specific retrieval parameters
TOP_K_INITIAL  = 10   # wider initial FAISS retrieval — gives graph filter more candidates
TOP_K_FINAL    = 5    # final context chunks after graph re-ranking (same as V0/V1)

# ── V2 graph constraint parameters
# Selected via 30-query ablation study:
#   PROXIMITY_BONUS tested: 0.1, 0.3, 0.5 → 0.3 maximised Context Recall
#   PENALTY tested: 0.25x, 0.5x, 0.75x   → 0.5x balanced precision and recall
HOP_LIMIT       = 2    # 2-hop neighbourhood: direct neighbours + their neighbours
PROXIMITY_BONUS = 0.3  # score bonus for chunks within HOP_LIMIT of the anchor node
PENALTY         = 0.5  # score multiplier for chunks outside HOP_LIMIT (penalise noise)

# ── Evaluation
N_QUERIES   = 50   # full evaluation — all 50 stratified queries
RANDOM_SEED = 42    # same seed as pilot notebooks — same query order

# ── Groq rate limit management (free tier: 30 req/min for Llama 3.1 8B)
# Each query makes ~3-4 LLM calls (generation + RAGAS metrics)
# 2s sleep between calls keeps usage safely under the limit
SLEEP_BETWEEN_CALLS = 12    # seconds between each Groq API call
RETRY_WAIT          = 60   # seconds to wait after a 429 rate-limit error
MAX_RETRIES         = 3    # max retries per call before marking query as failed

# ── Prompt — FIXED across V0, V1, V2
SYSTEM_PROMPT = (
    "You are a research assistant. Answer the question using ONLY the "
    "provided context. If the context does not contain enough information, "
    "say: I cannot answer this question from the provided context. "
    "Be concise and precise."
)

# ── File paths
# All artefacts must exist in data/ — built by the pilot notebooks
DATA_DIR          = "data"
QUERIES_FILE      = os.path.join(DATA_DIR, "queries.json")
V0_CHUNKS_FILE    = os.path.join(DATA_DIR, "v0_chunks.json")
V0_INDEX_FILE     = os.path.join(DATA_DIR, "v0_index.faiss")
V1_CHUNKS_FILE    = os.path.join(DATA_DIR, "v1_chunks.json")
V1_INDEX_FILE     = os.path.join(DATA_DIR, "v1_index.faiss")
GRAPH_FILE        = os.path.join(DATA_DIR, "v2_graph.pkl")

# Output files — one per variant, written incrementally
V0_RESULTS_FILE   = os.path.join(DATA_DIR, "full_eval_v0_results.json")
V1_RESULTS_FILE   = os.path.join(DATA_DIR, "full_eval_v1_results.json")
V2_RESULTS_FILE   = os.path.join(DATA_DIR, "full_eval_v2_results.json")
SUMMARY_FILE      = os.path.join(DATA_DIR, "full_eval_summary.json")

print("Configuration set.")
print(f"  Model:          {GROQ_MODEL}")
print(f"  Embed model:    {EMBED_MODEL}")
print(f"  Queries:        {N_QUERIES}")
print(f"  V0/V1 top-K:    {TOP_K_STANDARD}")
print(f"  V2 K initial:   {TOP_K_INITIAL}  →  K final: {TOP_K_FINAL}")
print(f"  V2 hop limit:   {HOP_LIMIT}")
print(f"  V2 bonus/pen:   +{PROXIMITY_BONUS} / x{PENALTY}")


## Cell 5 — Load All Artefacts

Loads the embedding model, queries, chunks, FAISS indexes, and graph built by the pilot notebooks.  
**Nothing is rebuilt here** — this notebook only runs evaluation against pre-built artefacts.


In [ ]:
import json
import random
from datasets import load_dataset

# ── Reload QASPER and regenerate queries.json with 150 queries
# This overwrites the 10-query pilot file

print("Loading QASPER to regenerate queries.json with 150 queries...")
ds     = load_dataset("allenai/qasper", revision="refs/convert/parquet", trust_remote_code=False)
papers = ds["train"]
print(f"Papers loaded: {len(papers)}")

def get_answer_type(answer):
    if answer.get("unanswerable", False): return "unanswerable"
    if answer.get("yes_no") is not None:  return "yes_no"
    if answer.get("extractive_spans"):    return "extractive"
    if answer.get("free_form_answer","").strip(): return "abstractive"
    return "unanswerable"

def extract_queries_from_paper(paper_id, paper):
    queries = []
    qas = paper.get("qas", {})
    if not isinstance(qas, dict): return queries
    for i, question in enumerate(qas.get("question", [])):
        question = question.strip()
        if not question: continue
        answers_wrapper = qas.get("answers", [])[i] if i < len(qas.get("answers", [])) else {}
        answers = answers_wrapper.get("answer", []) if isinstance(answers_wrapper, dict) else []
        if not answers: continue
        answer      = answers[0]
        answer_type = get_answer_type(answer)
        if answer_type == "unanswerable": continue
        yes_no    = answer.get("yes_no")
        free_form = answer.get("free_form_answer", "").strip()
        ext_spans = answer.get("extractive_spans", [])
        evidence  = answer.get("evidence", [])
        if answer_type == "yes_no":          answer_string = "Yes" if yes_no else "No"
        elif ext_spans:                       answer_string = " ".join(ext_spans).strip()
        else:                                 answer_string = free_form
        if not answer_string: continue
        queries.append({"paper_id": paper_id, "question": question,
                        "answer": answer_string, "evidence": evidence,
                        "answer_type": answer_type})
    return queries

# Extract all queries
all_queries = []
for paper in papers:
    all_queries.extend(extract_queries_from_paper(paper.get("id","unknown"), paper))
print(f"Total answerable queries: {len(all_queries)}")

# Stratified sample — 50% extractive, 30% abstractive, 20% yes_no, seed=42
def stratified_sample(all_queries, n, seed):
    rng     = random.Random(seed)
    buckets = {"extractive": [], "abstractive": [], "yes_no": []}
    for q in all_queries:
        if q["answer_type"] in buckets:
            buckets[q["answer_type"]].append(q)
    for b in buckets.values(): rng.shuffle(b)
    targets = {"extractive": int(n*0.50), "abstractive": int(n*0.30), "yes_no": int(n*0.20)}
    sampled, used = [], set()
    for t, target in targets.items():
        for q in buckets[t][:min(target, len(buckets[t]))]:
            sampled.append(q); used.add(q["question"])
    for bucket in sorted(buckets.values(), key=len, reverse=True):
        for q in bucket:
            if len(sampled) >= n: break
            if q["question"] not in used:
                sampled.append(q); used.add(q["question"])
    sampled = sampled[:n]; rng.shuffle(sampled)
    return sampled

queries = stratified_sample(all_queries, 150, 42)

# Save — overwrites the 10-query pilot file
with open(os.path.join(DATA_DIR, "queries.json"), "w") as f:
    json.dump(queries, f, indent=2)

# Verify
type_counts = {}
for q in queries:
    type_counts[q["answer_type"]] = type_counts.get(q["answer_type"], 0) + 1
print(f"\nSaved 150 queries → queries.json")
print(f"Breakdown: {type_counts}")

In [ ]:
# Validate all required artefact files exist before loading anything
required_files = [
    QUERIES_FILE, V0_CHUNKS_FILE, V0_INDEX_FILE,
    V1_CHUNKS_FILE, V1_INDEX_FILE, GRAPH_FILE
]
missing = [f for f in required_files if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(
        f"Missing artefacts — run the pilot notebooks first:\n"
        + "\n".join(f"  {f}" for f in missing)
    )
print("All artefact files found.\n")

# Load embedding model — used by all three variants for query embedding
print(f"Loading embedding model: {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)
print(f"  Dimension: {embed_model.get_sentence_embedding_dimension()}")

# Load queries — same 150 queries for all variants (ensures fair comparison)
print(f"\nLoading queries from {QUERIES_FILE}...")
with open(QUERIES_FILE) as f:
    queries = json.load(f)
print(f"  Queries loaded: {len(queries)}")

# Validate query count
if len(queries) < N_QUERIES:
    raise ValueError(
        f"queries.json has {len(queries)} queries but N_QUERIES={N_QUERIES}.\n"
        "Re-run the query sampling cell in v0_naive_rag.ipynb."
    )

# Show query type distribution as a sanity check
type_counts = {}
for q in queries[:N_QUERIES]:
    type_counts[q["answer_type"]] = type_counts.get(q["answer_type"], 0) + 1
print(f"  Query type breakdown: {type_counts}")

# Load V0 artefacts (fixed-window chunks + index)
print(f"\nLoading V0 artefacts...")
with open(V0_CHUNKS_FILE) as f:
    v0_chunks = json.load(f)
v0_index = faiss.read_index(V0_INDEX_FILE)
print(f"  V0 chunks: {len(v0_chunks):,}  |  V0 index vectors: {v0_index.ntotal:,}")

# Load V1 artefacts (paragraph-boundary chunks + index)
# V2 reuses these — graph is built on top of V1 chunk structure
print(f"\nLoading V1 artefacts...")
with open(V1_CHUNKS_FILE) as f:
    v1_chunks = json.load(f)
v1_index = faiss.read_index(V1_INDEX_FILE)
print(f"  V1 chunks: {len(v1_chunks):,}  |  V1 index vectors: {v1_index.ntotal:,}")

# Load V2 document graph
print(f"\nLoading V2 graph from {GRAPH_FILE}...")
with open(GRAPH_FILE, "rb") as f:
    G = pickle.load(f)
print(f"  Nodes: {G.number_of_nodes():,}  |  Edges: {G.number_of_edges():,}")
print(f"  Paragraph nodes: {len(v1_chunks):,}")
print(f"  Virtual section nodes: {G.number_of_nodes() - len(v1_chunks):,}")

print("\nAll artefacts loaded successfully.")


## Cell 6 — Groq Generation Function

Replaces `ollama.chat()` from the pilot notebooks with the Groq REST API.  
Uses Llama 3.1 8B instead of Llama 3.2 3B — consistent with the planned HPC evaluation model.


In [ ]:
# Initialise the Groq client once — reused for all generation calls
# Reads GROQ_API_KEY from the environment (loaded from .env in Cell 2)
groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))


def generate_with_groq(query: str, context_chunks: list) -> str:
    """
    Generate an answer using Llama 3.1 8B via the Groq API.

    Concatenates retrieved chunks with a clear separator so the LLM can
    distinguish where one passage ends and the next begins.

    Retries on 429 rate-limit errors up to MAX_RETRIES times.
    Returns an error string if all retries are exhausted — the query
    is then marked as failed in the results rather than crashing the loop.
    """
    context = "\n\n---\n\n".join(context_chunks)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"},
    ]

    for attempt in range(MAX_RETRIES):
        try:
            # Sleep before every Groq call to stay under the 30 req/min rate limit
            time.sleep(SLEEP_BETWEEN_CALLS)

            response = groq_client.chat.completions.create(
                model       = GROQ_MODEL,
                messages    = messages,
                temperature = TEMPERATURE,
                max_tokens  = 512,  # answers should be concise
            )
            return response.choices[0].message.content.strip()

        except Exception as e:
            error_str = str(e)
            if "429" in error_str or "rate_limit" in error_str.lower():
                # Rate limit hit — wait and retry
                print(f"      [rate limit] attempt {attempt+1}/{MAX_RETRIES} — waiting {RETRY_WAIT}s")
                time.sleep(RETRY_WAIT)
            else:
                print(f"      [generation error] {type(e).__name__}: {error_str[:100]}")
                return ""

    return "[generation failed after max retries]"


# ── Quick smoke test — one call to confirm the API key works
print("Smoke test — one Groq call...")
test_response = generate_with_groq(
    "What is retrieval-augmented generation?",
    ["RAG combines a retrieval system with a language model to ground responses in external documents."]
)
print(f"Response: {test_response[:150]}")
print("\nGroq generation working.")


## Cell 7 — RAGAS Evaluation Setup

Configures RAGAS to use ChatGroq as the judge LLM instead of ChatOllama.  
The same Llama 3.1 8B model is used for both generation (Cell 6) and evaluation — consistent with the methodology.


In [ ]:
# Wrap ChatGroq for LangChain compatibility
# RAGAS requires a LangChain LLM interface for its internal metric scoring
ragas_llm = LangchainLLMWrapper(
    ChatGroq(
        model       = GROQ_MODEL,
        temperature = TEMPERATURE,
        api_key     = os.environ.get("GROQ_API_KEY"),
    )
)

# RAGAS also needs an embedding model for answer_relevancy scoring
# Using the same embedding model as the retrieval pipeline for consistency
ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name=EMBED_MODEL)
)

# RunConfig: generous timeout because Groq occasionally has slow responses under load
# raise_exceptions=False in evaluate() prevents a single bad JSON response
# from crashing the entire evaluation loop — safe_score() handles NaN values
ragas_run_config = RunConfig(
    timeout     = 120,  # seconds per metric call
    max_retries = 2,
    max_wait    = 30,
)

print("RAGAS configured.")
print(f"  Judge LLM:   {GROQ_MODEL} via Groq")
print(f"  Embeddings:  {EMBED_MODEL}")


def safe_score(result_dict: dict, key: str) -> float:
    """
    Safely extract a RAGAS metric score.

    RAGAS returns NaN when the LLM response cannot be parsed as valid JSON.
    This is a known issue with smaller models and occasional API timeouts.
    This wrapper replaces NaN with 0.0 rather than propagating it.

    A 0.0 from a parse failure is flagged via the 'status' field in the
    result record — it can be filtered in post-processing if needed.
    """
    val = result_dict.get(key)
    if val is None:
        return 0.0
    score = val[0] if isinstance(val, list) else val
    try:
        f = float(score)
        return 0.0 if f != f else f  # f != f catches NaN
    except (TypeError, ValueError):
        return 0.0


def evaluate_with_ragas(question: str, answer: str, ground_truth: str, retrieved: list) -> dict:
    """
    Run RAGAS evaluation for a single query result.

    Evaluates four metrics:
      context_recall    — was the right evidence retrieved?
      context_precision — was retrieved content usefully ranked?
      faithfulness      — is the answer grounded in the retrieved context?
      answer_relevancy  — does the answer address the question?

    A small extra sleep is added before evaluation to absorb the
    multiple internal API calls RAGAS makes per metric.
    """
    # Extra sleep before RAGAS block — it makes 2-3 additional Groq calls
    time.sleep(SLEEP_BETWEEN_CALLS)

    ragas_dataset = Dataset.from_dict({
        "question":      [question],
        "answer":        [answer],
        "contexts":      [retrieved],
        "ground_truth":  [ground_truth],
        "ground_truths": [[ground_truth]],
    })

    try:
        result = evaluate(
            ragas_dataset,
            metrics          = [context_recall, context_precision, faithfulness, answer_relevancy],
            llm              = ragas_llm,
            embeddings       = ragas_embeddings,
            raise_exceptions = False,  # do not crash on bad JSON — use safe_score
            run_config       = ragas_run_config,
        )
        return {
            "context_recall":    safe_score(result, "context_recall"),
            "context_precision": safe_score(result, "context_precision"),
            "faithfulness":      safe_score(result, "faithfulness"),
            "answer_relevancy":  safe_score(result, "answer_relevancy"),
        }
    except Exception as e:
        print(f"      [ragas error] {type(e).__name__}: {str(e)[:100]}")
        return {k: 0.0 for k in ["context_recall", "context_precision",
                                   "faithfulness", "answer_relevancy"]}

print("\nevaluate_with_ragas and safe_score defined.")


## Cell 8 — Retrieval Functions (V0, V1, V2)

All three retrieval functions are defined here.  
V0 and V1 are identical in mechanism — the only difference is which FAISS index they query (built from different chunks).  
V2 adds the graph proximity filter on top of V1's chunks.


In [ ]:
# ── V0 RETRIEVAL — Naive RAG
# Pure cosine similarity over fixed-window chunks.
# No structural awareness — this is the failure condition baseline.

def retrieve_v0(query: str) -> list:
    """
    V0: embed query, search V0 FAISS index, return TOP_K_STANDARD chunk texts.

    FAISS IndexFlatL2 on L2-normalised vectors is equivalent to cosine
    similarity ranking. The top-5 results are returned with no re-ranking.
    """
    q_vec = embed_model.encode(
        [query],
        convert_to_numpy     = True,
        normalize_embeddings = True,  # must match how chunks were embedded
    ).astype("float32")

    _, indices = v0_index.search(q_vec, TOP_K_STANDARD)
    return [v0_chunks[idx]["chunk_text"] for idx in indices[0] if idx != -1]


# ── V1 RETRIEVAL — Structure-Aware Chunking
# Same mechanism as V0 but queries the V1 FAISS index (paragraph-boundary chunks).
# Any score difference vs V0 is attributable to chunking strategy only.

def retrieve_v1(query: str) -> list:
    """
    V1: identical retrieval mechanism to V0 but operates on paragraph chunks.

    The V1 FAISS index was built from paragraph-boundary chunks (v1_chunks.json)
    rather than fixed-window chunks (v0_chunks.json). Everything else is unchanged.
    """
    q_vec = embed_model.encode(
        [query],
        convert_to_numpy     = True,
        normalize_embeddings = True,
    ).astype("float32")

    _, indices = v1_index.search(q_vec, TOP_K_STANDARD)
    return [v1_chunks[idx]["chunk_text"] for idx in indices[0] if idx != -1]


# ── V2 RETRIEVAL — Graph-Constrained
# Taken directly from v2_graph_constrained_retrieval.ipynb Cell 7.
# Adds graph proximity filter on top of V1's paragraph chunks.

def retrieve_v2(query: str) -> tuple:
    """
    V2 graph-constrained retrieval.

    Step 1 — Widened initial retrieval: fetch TOP_K_INITIAL=10 candidates.
    Step 2 — Anchor selection: top-1 by cosine similarity = anchor node.
    Step 3 — Graph re-scoring: boost chunks within HOP_LIMIT of anchor,
              penalise chunks outside HOP_LIMIT.
    Step 4 — Adjacency expansion: pull in sequential graph neighbours of
              anchor not already in the candidate set.
    Step 5 — Re-rank by adjusted score, return TOP_K_FINAL=5.

    Returns:
        retrieved_texts — list of chunk texts for the LLM prompt
        provenance      — metadata list for Provenance Coverage metric
    """
    q_vec = embed_model.encode(
        [query],
        convert_to_numpy     = True,
        normalize_embeddings = True,
    ).astype("float32")

    distances, indices = v1_index.search(q_vec, TOP_K_INITIAL)

    # Convert L2 distances on normalised vectors to cosine similarity
    # For unit vectors: cosine_sim = 1 - (L2_dist^2 / 2)
    candidates = []
    for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        if idx == -1:
            continue
        cosine_sim = float(1 - dist / 2)
        candidates.append({
            "chunk_id":        v1_chunks[idx]["chunk_id"],
            "chunk_idx":       idx,
            "chunk_text":      v1_chunks[idx]["chunk_text"],
            "paper_id":        v1_chunks[idx]["paper_id"],
            "section_name":    v1_chunks[idx]["section_name"],
            "paragraph_index": v1_chunks[idx]["paragraph_index"],
            "cosine_sim":      cosine_sim,
            "rank":            rank,
        })

    if not candidates:
        return [], []

    # Anchor = top-1 candidate by cosine similarity
    anchor_id = candidates[0]["chunk_id"]

    # Re-score each candidate based on graph distance to anchor
    rescored = []
    for c in candidates:
        score      = c["cosine_sim"]
        graph_dist = None

        if c["chunk_id"] == anchor_id:
            graph_dist = 0  # anchor scores itself
        elif G.has_node(c["chunk_id"]) and G.has_node(anchor_id):
            try:
                path_len   = nx.shortest_path_length(G, anchor_id, c["chunk_id"])
                graph_dist = path_len
                if path_len <= HOP_LIMIT:
                    score = score + PROXIMITY_BONUS  # structurally close — boost
                else:
                    score = score * PENALTY          # structurally distant — penalise
            except nx.NetworkXNoPath:
                graph_dist = None
                score      = score * PENALTY         # no structural connection — penalise
        else:
            score = score * PENALTY

        rescored.append({**c, "final_score": score, "graph_dist": graph_dist})

    # Adjacency expansion — pull in sequential graph neighbours of anchor
    # that cosine similarity did not rank in the initial top-10
    existing_ids = {c["chunk_id"] for c in rescored}

    if G.has_node(anchor_id):
        for neighbour_id in G.neighbors(anchor_id):
            if neighbour_id in existing_ids:
                continue
            edge_data = G.get_edge_data(anchor_id, neighbour_id)
            if edge_data and edge_data.get("type") == "sequential":
                neighbour_chunks = [c for c in v1_chunks if c["chunk_id"] == neighbour_id]
                if neighbour_chunks:
                    nc     = neighbour_chunks[0]
                    nc_vec = embed_model.encode(
                        [nc["chunk_text"]],
                        convert_to_numpy=True, normalize_embeddings=True,
                    ).astype("float32")
                    nc_dist = float(np.linalg.norm(q_vec - nc_vec))
                    nc_sim  = float(1 - nc_dist**2 / 2)
                    rescored.append({
                        "chunk_id":        nc["chunk_id"],
                        "chunk_idx":       nc["chunk_id"],
                        "chunk_text":      nc["chunk_text"],
                        "paper_id":        nc["paper_id"],
                        "section_name":    nc["section_name"],
                        "paragraph_index": nc["paragraph_index"],
                        "cosine_sim":      nc_sim,
                        "rank":            999,
                        "final_score":     nc_sim + PROXIMITY_BONUS,
                        "graph_dist":      1,
                    })

    # Sort by final score and return top-K
    rescored.sort(key=lambda x: x["final_score"], reverse=True)
    final = rescored[:TOP_K_FINAL]

    retrieved_texts = [c["chunk_text"] for c in final]
    provenance      = [{
        "chunk_id":        c["chunk_id"],
        "paper_id":        c["paper_id"],
        "section_name":    c["section_name"],
        "paragraph_index": c["paragraph_index"],
        "cosine_sim":      round(c["cosine_sim"], 4),
        "graph_dist":      c["graph_dist"],
        "final_score":     round(c["final_score"], 4),
    } for c in final]

    return retrieved_texts, provenance


def compute_provenance_coverage(provenance: list) -> float:
    """
    Provenance Coverage — V2-only supplementary metric.

    Measures what proportion of the final retrieved chunks have a traceable
    structural path to the anchor node via the document graph.

    Score = chunks with graph_dist not None / total chunks retrieved

    This directly quantifies the graph constraint's effectiveness at
    surfacing structurally connected evidence.
    Cannot be computed for V0 or V1 — neither uses a graph.
    """
    if not provenance:
        return 0.0
    with_path = sum(1 for p in provenance if p["graph_dist"] is not None)
    return round(with_path / len(provenance), 4)


# Quick smoke test on all three retrieve functions
print("Smoke testing retrieval functions...")
test_q = queries[0]["question"]
print(f"  Question: {test_q[:60]}...")

r0 = retrieve_v0(test_q)
print(f"  V0: retrieved {len(r0)} chunks")

r1 = retrieve_v1(test_q)
print(f"  V1: retrieved {len(r1)} chunks")

r2, prov = retrieve_v2(test_q)
pc = compute_provenance_coverage(prov)
print(f"  V2: retrieved {len(r2)} chunks  |  provenance coverage: {pc:.2f}")
print("\nAll retrieval functions working.")


## Cell 9 — Incremental Save and Resume Helpers

Results are saved after every query. If the run is interrupted at query N, re-run the notebook and it resumes from query N+1 automatically.


In [ ]:
def load_existing_results(filepath: str) -> list:
    """
    Load previously saved results for resume support.
    Returns empty list if file does not exist (fresh run).
    """
    if os.path.exists(filepath):
        with open(filepath) as f:
            return json.load(f)
    return []


def save_results(results: list, filepath: str):
    """
    Write results list to disk as JSON.
    Called after every query — crash-safe incremental save.
    """
    with open(filepath, "w") as f:
        json.dump(results, f, indent=2)


print("Save/resume helpers defined.")


## Cell 10 — Single Query Runner

Encapsulates the full retrieve → generate → evaluate pipeline for one query.  
Used by all three variants — variant-specific behaviour is handled by passing the correct retrieve function.


In [ ]:
def run_single_query(query_record: dict, query_index: int,
                     retrieve_fn, variant_name: str) -> dict:
    """
    Run one query through the full pipeline: retrieve → generate → evaluate.

    Returns a result dict containing:
      - Query metadata (index, paper_id, question, answer_type)
      - The generated answer
      - RAGAS metric scores (context_recall, context_precision,
        faithfulness, answer_relevancy)
      - Provenance Coverage (V2 only)
      - Timing breakdown (retrieve, generate, evaluate, total)
      - status: 'success' or 'failed'
    """
    t_start = time.time()

    result = {
        "query_index":  query_index,
        "variant":      variant_name,
        "paper_id":     query_record.get("paper_id"),
        "question":     query_record["question"],
        "ground_truth": query_record["answer"],
        "answer_type":  query_record.get("answer_type"),
        "timestamp":    datetime.utcnow().isoformat(),
    }

    try:
        # ── Step 1: Retrieval
        t0 = time.time()
        if variant_name == "V2":
            retrieved_texts, provenance = retrieve_fn(query_record["question"])
            result["provenance_coverage"] = compute_provenance_coverage(provenance)
        else:
            retrieved_texts = retrieve_fn(query_record["question"])
        t_retrieve = time.time() - t0

        if not retrieved_texts:
            result["status"] = "failed"
            result["error"]  = "retrieval returned empty set"
            result.update({k: 0.0 for k in ["context_recall", "context_precision",
                                              "faithfulness", "answer_relevancy"]})
            return result

        # ── Step 2: Generation via Groq
        t0 = time.time()
        generated_answer = generate_with_groq(query_record["question"], retrieved_texts)
        t_generate = time.time() - t0

        if not generated_answer or "failed after max retries" in generated_answer:
            result["status"]           = "failed"
            result["error"]            = "generation failed"
            result["generated_answer"] = generated_answer
            result.update({k: 0.0 for k in ["context_recall", "context_precision",
                                              "faithfulness", "answer_relevancy"]})
            return result

        result["generated_answer"] = generated_answer

        # ── Step 3: RAGAS evaluation
        t0 = time.time()
        metrics = evaluate_with_ragas(
            question     = query_record["question"],
            answer       = generated_answer,
            ground_truth = query_record["answer"],
            retrieved    = retrieved_texts,
        )
        t_evaluate = time.time() - t0

        result.update(metrics)
        result["timing"] = {
            "retrieve_s":  round(t_retrieve, 2),
            "generate_s":  round(t_generate, 2),
            "evaluate_s":  round(t_evaluate, 2),
            "total_s":     round(time.time() - t_start, 2),
        }
        result["status"] = "success"

    except Exception as e:
        result["status"]    = "failed"
        result["error"]     = f"{type(e).__name__}: {str(e)[:200]}"
        result["traceback"] = traceback.format_exc()[-500:]
        result.update({k: 0.0 for k in ["context_recall", "context_precision",
                                          "faithfulness", "answer_relevancy"]})

    return result

print("run_single_query defined.")


## Cell 11 — Variant Runner

Runs all 150 queries for a single variant with resume support and incremental saving.


In [ ]:
def run_variant(variant_name: str, retrieve_fn, results_file: str) -> list:
    """
    Run the full 150-query evaluation for one pipeline variant.

    - Loads any previously saved results (resume support)
    - Skips queries that already have a successful result
    - Saves results incrementally after every query
    - Prints per-query progress in real time
    """
    print(f"\n{'='*65}")
    print(f"  VARIANT: {variant_name}")
    print(f"  Output:  {results_file}")
    print(f"{'='*65}")

    results           = load_existing_results(results_file)
    completed_indices = {r["query_index"] for r in results if r.get("status") == "success"}
    print(f"  Already completed: {len(completed_indices)} / {N_QUERIES}")

    t_variant_start = time.time()

    for i, q in enumerate(queries[:N_QUERIES]):

        # Skip queries that already have a successful result
        if i in completed_indices:
            continue

        print(f"\n  [{variant_name}] {i+1}/{N_QUERIES} | {q.get('answer_type',''):12} | "
              f"{q['question'][:55]}...")

        result = run_single_query(
            query_record = q,
            query_index  = i,
            retrieve_fn  = retrieve_fn,
            variant_name = variant_name,
        )

        results.append(result)
        save_results(results, results_file)  # save after every query — crash safe

        # Print per-query summary line
        if result["status"] == "success":
            cr  = result.get("context_recall", 0.0)
            cp  = result.get("context_precision", 0.0)
            f   = result.get("faithfulness", 0.0)
            ar  = result.get("answer_relevancy", 0.0)
            pc  = result.get("provenance_coverage", "N/A")
            t   = result.get("timing", {}).get("total_s", 0)
            pc_str = f"{pc:.2f}" if isinstance(pc, float) else "N/A"
            print(f"    ✓ CR={cr:.3f}  CP={cp:.3f}  F={f:.3f}  AR={ar:.3f}  "
                  f"PC={pc_str}  [{t:.0f}s]")
        else:
            print(f"    ✗ FAILED: {result.get('error', 'unknown')[:80]}")

    successful = [r for r in results if r.get("status") == "success"]
    elapsed    = time.time() - t_variant_start
    print(f"\n  {variant_name} complete: {len(successful)}/{N_QUERIES} successful "
          f"in {elapsed/60:.1f} min")

    return results

print("run_variant defined.")


## Cell 12 — Run V0 (Naive RAG Baseline)

**What changes:** Fixed-window chunking (256 tokens, 32 overlap) + cosine similarity retrieval  
**What is fixed:** Everything else  
**Purpose:** Establishes the performance floor — this is the failure condition


In [ ]:
v0_results = run_variant(
    variant_name = "V0",
    retrieve_fn  = retrieve_v0,
    results_file = V0_RESULTS_FILE,
)


## Cell 13 — Run V1 (Structure-Aware Chunking)

**What changes:** Paragraph-boundary chunking (one paragraph = one chunk)  
**What is fixed:** Embedding model, FAISS index type, cosine similarity retrieval, LLM, prompt  
**Purpose:** Isolates the chunking effect — any score difference vs V0 is from chunking alone


In [ ]:
v1_results = run_variant(
    variant_name = "V1",
    retrieve_fn  = retrieve_v1,
    results_file = V1_RESULTS_FILE,
)


## Cell 14 — Run V2 (Graph-Constrained Retrieval)

**What changes:** Graph proximity filter over V1 paragraph chunks  
**Graph parameters:** HOP_LIMIT=2, PROXIMITY_BONUS=+0.3, PENALTY=×0.5  
**What is fixed:** Embedding model, LLM, prompt, chunking strategy (same as V1)  
**Purpose:** Isolates the graph constraint effect — any score difference vs V1 is from the graph alone


In [ ]:
v2_results = run_variant(
    variant_name = "V2",
    retrieve_fn  = retrieve_v2,
    results_file = V2_RESULTS_FILE,
)


## Cell 15 — Aggregate Results and Summary Table

Computes mean ± std for each metric across all successful queries per variant.  
This is the data that populates Section 5 (Results) of the dissertation report.


In [ ]:
def aggregate_metrics(results: list, variant_name: str) -> dict:
    """
    Compute mean and standard deviation for each RAGAS metric
    across all successful queries.

    Filters to status=success only so failed queries (zeroed scores)
    do not distort the mean. The n_failed count is reported separately
    so the examiner can assess data quality.
    """
    successful = [r for r in results if r.get("status") == "success"]
    failed     = [r for r in results if r.get("status") != "success"]

    if not successful:
        return {"variant": variant_name, "n_successful": 0, "n_failed": len(failed)}

    metrics = ["context_recall", "context_precision", "faithfulness",
               "answer_relevancy", "provenance_coverage"]

    agg = {
        "variant":     variant_name,
        "n_successful": len(successful),
        "n_failed":     len(failed),
        "n_total":      len(results),
    }

    for m in metrics:
        values = [r[m] for r in successful if m in r and r[m] is not None]
        if values:
            arr = np.array(values, dtype=float)
            agg[f"{m}_mean"] = round(float(np.mean(arr)), 4)
            agg[f"{m}_std"]  = round(float(np.std(arr)),  4)
            agg[f"{m}_min"]  = round(float(np.min(arr)),  4)
            agg[f"{m}_max"]  = round(float(np.max(arr)),  4)

    # Per answer-type breakdown — useful for Section 6 analysis
    for answer_type in ["extractive", "abstractive", "yes_no"]:
        subset = [r for r in successful if r.get("answer_type") == answer_type]
        if subset:
            for m in ["context_recall", "faithfulness"]:
                values = [r[m] for r in subset if m in r]
                if values:
                    agg[f"{answer_type}_{m}_mean"] = round(float(np.mean(values)), 4)

    return agg


# Aggregate all three variants
v0_agg = aggregate_metrics(v0_results, "V0")
v1_agg = aggregate_metrics(v1_results, "V1")
v2_agg = aggregate_metrics(v2_results, "V2")

# Save summary
summary = {
    "V0": v0_agg,
    "V1": v1_agg,
    "V2": v2_agg,
    "metadata": {
        "model":              GROQ_MODEL,
        "embed_model":        EMBED_MODEL,
        "n_queries":          N_QUERIES,
        "v0_chunk_size":      256,
        "v0_chunk_overlap":   32,
        "v1_chunking":        "paragraph_boundary",
        "v2_top_k_initial":   TOP_K_INITIAL,
        "v2_top_k_final":     TOP_K_FINAL,
        "v2_hop_limit":       HOP_LIMIT,
        "v2_proximity_bonus": PROXIMITY_BONUS,
        "v2_penalty":         PENALTY,
        "completed_at":       datetime.utcnow().isoformat(),
    }
}

with open(SUMMARY_FILE, "w") as f:
    json.dump(summary, f, indent=2)
print(f"Summary saved → {SUMMARY_FILE}")

# ── Print the comparison table
print(f"\n{'='*70}")
print("  FULL EVALUATION RESULTS — 150 QUERIES")
print(f"{'='*70}")
print(f"  {'Metric':<26} {'V0':>8} {'V1':>8} {'V2':>8}")
print(f"  {'-'*58}")

metrics_display = [
    ("Context Recall",     "context_recall_mean"),
    ("Context Precision",  "context_precision_mean"),
    ("Faithfulness",       "faithfulness_mean"),
    ("Answer Relevancy",   "answer_relevancy_mean"),
    ("Provenance Coverage","provenance_coverage_mean"),
]

for label, key in metrics_display:
    def fmt(d, k):
        v = d.get(k)
        return f"{v:.4f}" if isinstance(v, float) else "N/A"
    print(f"  {label:<26} {fmt(v0_agg,key):>8} {fmt(v1_agg,key):>8} {fmt(v2_agg,key):>8}")

print(f"\n  Successful queries:")
print(f"    V0: {v0_agg.get('n_successful',0)}/{v0_agg.get('n_total',0)}")
print(f"    V1: {v1_agg.get('n_successful',0)}/{v1_agg.get('n_total',0)}")
print(f"    V2: {v2_agg.get('n_successful',0)}/{v2_agg.get('n_total',0)}")

# ── Delta table — shows the change at each step
# V0→V1 = chunking effect, V1→V2 = graph effect, V0→V2 = combined
print(f"\n  {'Delta analysis':<26} {'V0→V1':>8} {'V1→V2':>8} {'V0→V2':>8}")
print(f"  {'-'*58}")
for label, key in metrics_display[:4]:
    v0_val = v0_agg.get(key)
    v1_val = v1_agg.get(key)
    v2_val = v2_agg.get(key)
    if all(isinstance(v, float) for v in [v0_val, v1_val, v2_val]):
        d01 = v1_val - v0_val
        d12 = v2_val - v1_val
        d02 = v2_val - v0_val
        print(f"  {label:<26} {d01:>+8.4f} {d12:>+8.4f} {d02:>+8.4f}")

print(f"{'='*70}")


In [ ]:
from langchain_groq import ChatGroq
from ragas.llms import LangchainLLMWrapper

# Replace the judge with mixtral — much more reliable at structured JSON
ragas_llm = LangchainLLMWrapper(
    ChatGroq(
        model   = "mixtral-8x7b-32768",
        temperature = 0,
        api_key = os.environ.get("GROQ_API_KEY"),
    )
)
print("RAGAS judge switched to Mixtral")

In [ ]:
q = queries[0]
retrieved = retrieve_v0(q["question"])
generated = generate_with_groq(q["question"], retrieved)
metrics   = evaluate_with_ragas(q["question"], generated,
                                 q["answer"], retrieved)
print(metrics)